In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, trim, lower

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
df = spark.read.parquet("/user/hive/warehouse/sclsepb4532_raw.db/orders", header=True, inferSchema=True)

26/11/05 18:21:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/11/05 18:21:54 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/11/05 18:21:54 WARN Utils: Serv

In [2]:
df = df.filter("total_amount >= 0")

In [3]:
df = df.withColumn("order_status", trim(col("order_status")))
valid_statuses = ["Delivered","Shipped","Cancelled","Placed"]

# Keep only rows with valid order_status values
df = df.filter(col("order_status").isin(valid_statuses))

In [4]:
df = df.filter(col("order_date").isNotNull() &(col("order_date") != ""))

In [5]:
df = df.withColumn("order_date", to_timestamp(col("order_date")))

In [6]:
df = df.filter(lower(col("order_status")) != "cancelled")

In [7]:
df.write.mode("overwrite").saveAsTable("sclsepb4532_staging.orders_stg")

26/11/05 18:22:47 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
